# S28A — queue_ahead Sweep (execution sensitivity)
Single axis: queue_ahead. Delta, TTL, qty, lambda frozen. Layer A only (no PnL/economics).
Spec: OBSIDIAN/docs/superpowers/specs/2026-05-30-het-sweep-s28a-design.md

In [ ]:
# --- repo-root bootstrap ---
# Notebook uses absolute imports (research.experiments.*) and repo-relative paths
# (data/raw/..., research/experiments/outputs). Anchor CWD + sys.path to repo root
# so it runs regardless of where Jupyter's kernel was launched. Idempotent.
import sys, os
from pathlib import Path

_root = Path.cwd()
while not (_root / "research" / "__init__.py").exists() and _root != _root.parent:
    _root = _root.parent
if not (_root / "research" / "__init__.py").exists():
    raise RuntimeError("repo root not found: launch Jupyter from within the TRADING-BOT repo")
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("repo root:", _root)

In [ ]:
import json, hashlib
from pathlib import Path
from decimal import Decimal
import numpy as np, matplotlib.pyplot as plt
from research.experiments.het_sweep import (
    load_canonical_tape, generate_passive_orders, run_queue_ahead_sweep, summarize_sweep,
    SEED_EXPERIMENT, LAMBDA_PER_SEC, SIDE_SEED, TTL_FROZEN_MS, QUEUE_AHEAD_GRID_FROZEN,
)
from live.order_types import OrderSide

TAPE_PATH = "data/raw/BTCUSDT_AGGTRADES.csv"
FINAL_TTL_MS = TTL_FROZEN_MS                             # 60s, frozen by S28A-0 (Caso B); 300s/900s saturate the grid
FINAL_GRID = QUEUE_AHEAD_GRID_FROZEN                     # {0,0.5,2,5,15,50} BTC, spans P0->P90 @ 60s (P50=4.879, P90=55.860)
OUT = Path("research/experiments/outputs"); OUT.mkdir(parents=True, exist_ok=True)

CONFIG = {"delta_ticks": 5, "order_qty": "0.01", "lambda_per_sec": LAMBDA_PER_SEC,
          "seed": SEED_EXPERIMENT, "ttl_ms": FINAL_TTL_MS, "grid": [str(q) for q in FINAL_GRID]}
SPEC_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]
print("SPEC_HASH:", SPEC_HASH)

In [ ]:
tape = load_canonical_tape(TAPE_PATH)
orders = []
for side in (OrderSide.BUY, OrderSide.SELL):
    orders += generate_passive_orders(tape, side, LAMBDA_PER_SEC, FINAL_TTL_MS, seed=[SEED_EXPERIMENT, SIDE_SEED[side]])
print(f"orders: {len(orders):,}  (expect ~43k/side over the month)")
rows = run_queue_ahead_sweep(tape, orders, FINAL_TTL_MS, FINAL_GRID)   # window built once/order, swept across grid
summary = summarize_sweep(rows, FINAL_GRID)
for q in FINAL_GRID:
    s = summary[q]
    print(f"q={s['queue_ahead']:>4}  fill_rate={s['fill_rate']:.4f} "
          f"CI=[{s['fill_rate_ci'][0]:.4f},{s['fill_rate_ci'][1]:.4f}]  "
          f"reach={s['reach_rate']:.3f}  med_ttf={s['median_time_to_fill_ms']}")

In [ ]:
qs = [float(q) for q in FINAL_GRID]
fr = [summary[q]["fill_rate"] for q in FINAL_GRID]
lo = [summary[q]["fill_rate_ci"][0] for q in FINAL_GRID]
hi = [summary[q]["fill_rate_ci"][1] for q in FINAL_GRID]
ttf = [summary[q]["median_time_to_fill_ms"] or np.nan for q in FINAL_GRID]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
a1.errorbar(qs, fr, yerr=[np.subtract(fr, lo), np.subtract(hi, fr)], marker="o", capsize=4)
a1.set_xlabel("queue_ahead (BTC)"); a1.set_ylabel("fill_rate"); a1.set_title("HET: fill_rate(queue_ahead)")
a2.plot(qs, ttf, marker="s", color="C1")
a2.set_xlabel("queue_ahead (BTC)"); a2.set_ylabel("median time-to-fill (ms) | filled")
a2.set_title("HET: time-to-fill(queue_ahead)")
fig.savefig(OUT / f"S28A_het_curves_{SPEC_HASH}.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import csv
serial = {str(q): summary[q] for q in FINAL_GRID}
for v in serial.values():
    v["fill_rate_ci"] = list(v["fill_rate_ci"])
(OUT / f"S28A_summary_{SPEC_HASH}.json").write_text(
    json.dumps({"config": CONFIG, "spec_hash": SPEC_HASH, "summary": serial}, indent=2), encoding="utf-8")

with open(OUT / f"S28A_rows_{SPEC_HASH}.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["queue_ahead", "arrival_ts", "filled", "time_to_fill_ms", "reached",
                "queue_consumed", "queue_remaining", "through_volume"])
    for r in rows:
        w.writerow([str(r["queue_ahead"]), r["arrival_ts"], r["filled"], r["time_to_fill_ms"],
                    r["reached"], str(r["queue_consumed"]), str(r["queue_remaining"]), str(r["through_volume"])])
print("FROZEN artifacts written with SPEC_HASH", SPEC_HASH)

Read fill_rate(queue_ahead): if it decays then plateaus, queue_ahead matters (calibrate near the knee).
If fill_rate is flat ~1.0 and only median TTF / queue_consumed move, depth (Delta) dominates the queue
-> motivates S28B. Either outcome is a clean S28A result. NO PnL/economics here (Layer B, deferred).